# Track4World - 官方展示效果复现

**目标**: 完全按照官方README的推荐方式，实现前景-背景分离的世界坐标系可视化

**关键配置**:
- 坐标系: `world_depthanythingv3` (世界坐标系)
- 模型: DA3 (更好的深度估计)
- 分割: DINO + SAM2 (前景-背景分离)
- 可视化: `vis_3d_efep_world.py`

**使用流程**:
1. Cell 1: 检查GPU
2. Cell 2: 安装依赖
3. Cell 3: 下载权重 (Track4World + SAM2)
4. Cell 4: 上传视频
5. Cell 5: DINO+SAM2分割 (关键步骤！)
6. Cell 6: Track4World 3d_efep推理
7. Cell 7: 打包下载结果

In [ ]:
# Cell 1: 检查 GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
!nvidia-smi

In [ ]:
# Cell 2: 克隆仓库 + 安装依赖（完全按照官方方式）
import os, sys

_flag = "/content/.t4w_official_installed"

if not os.path.exists(_flag):
    print("=== 首次安装：克隆仓库 ===\n")

    # 克隆Track4World（不使用--recurse-submodules）
    if not os.path.exists("/content/Track4World"):
        !git clone https://github.com/TencentARC/Track4World.git /content/Track4World

    os.chdir("/content/Track4World")

    # 按照官方方式安装第三方模块
    print("\n=== 安装第三方模块（官方方式）===")
    
    # 1. Install utils3d
    if not os.path.exists("/content/utils3d"):
        !git clone https://github.com/jiah-cloud/utils3d.git /content/utils3d
    
    # 2. Setup Grounded-SAM-2
    if not os.path.exists("submodules"):
        !git clone https://github.com/IDEA-Research/Grounded-SAM-2.git submodules
        os.chdir("submodules")
        !pip install -q -e .
        !pip install -q --no-build-isolation -e grounding_dino
        os.chdir("/content/Track4World")

    # 修改requirements.txt避免版本冲突
    with open('requirements.txt', 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        if line.startswith('torch==') or line.startswith('torchvision==') or line.startswith('torchaudio=='):
            new_lines.append('#' + line)
        elif line.startswith('open3d==0.18.0'):
            new_lines.append('open3d>=0.19.0\n')
        else:
            new_lines.append(line)

    with open('requirements.txt', 'w') as f:
        f.writelines(new_lines)

    # 先固定numpy版本（关键！）
    print("\n=== 固定numpy版本 ===")
    !pip install -q numpy==1.26.4

    # 安装PyTorch
    !pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

    # 安装Track4World依赖
    !pip install -q -r requirements.txt

    # 强制重装opencv
    !pip install -q opencv-python==4.10.0.84 --force-reinstall

    # 安装其他依赖
    !pip install -q open3d viser plotly supervision

    open(_flag, "w").close()

    print("\n✓ 安装完成！正在重启运行时...")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

else:
    print("=== 重启后：配置环境 ===")
    
    if not os.path.exists("/content/Track4World"):
        print("❌ Track4World目录不存在，请先删除标志文件：")
        print("!rm -f /content/.t4w_official_installed")
        raise FileNotFoundError("Track4World目录不存在")
    
    os.chdir("/content/Track4World")
    
    import sys
    sys.path.insert(0, '/content/utils3d')
    sys.path.insert(0, '/content/Track4World')
    sys.path.insert(0, '/content/Track4World/submodules')

    print("\n=== 依赖检查 ===")
    for pkg, mod in [
        ('torch','torch'), ('numpy','numpy'), ('cv2','cv2'),
        ('open3d','open3d'), ('viser','viser'), ('supervision','supervision'),
    ]:
        try:
            m = __import__(mod)
            print(f"  ✓ {pkg}: {getattr(m,'__version__','?')}")
        except ImportError:
            print(f"  ✗ {pkg}: 未安装")

    print("\n✓ 环境就绪，可继续运行 Cell 3")

In [ ]:
# Cell 3: 下载权重
import os
os.chdir("/content/Track4World")
os.makedirs("checkpoints", exist_ok=True)

# 下载Track4World权重
weights = [
    ("track4world_da3.pth", "https://huggingface.co/TencentARC/Track4World/resolve/main/track4world_da3.pth"),
    ("track4world_moge.pth", "https://huggingface.co/TencentARC/Track4World/resolve/main/track4world_moge.pth"),
]

for name, url in weights:
    path = f"checkpoints/{name}"
    if not os.path.exists(path):
        print(f"下载 {name}...")
        !wget -q --show-progress -O {path} {url}
    else:
        print(f"✓ {name} 已存在")

# 下载SAM2权重
sam2_path = "checkpoints/sam2.1_hiera_large.pt"
if not os.path.exists(sam2_path):
    print("\n下载 SAM2 权重...")
    !wget -q --show-progress -O {sam2_path} https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
else:
    print("✓ SAM2 权重已存在")

print("\n=== 权重文件 ===")
!ls -lh checkpoints/*.pt checkpoints/*.pth 2>/dev/null | awk '{print $9, $5}'

In [ ]:
# Cell 4: 上传视频（支持自动检测已上传视频）
from google.colab import files
import os

os.chdir("/content/Track4World")
os.makedirs("input_videos", exist_ok=True)

# 检查是否已有视频
existing_videos = [f for f in os.listdir("input_videos") if f.endswith('.mp4')]

if existing_videos:
    print(f"检测到已上传的视频：")
    for i, fname in enumerate(existing_videos):
        size_mb = os.path.getsize(f"input_videos/{fname}") / 1e6
        print(f"  {i+1}. {fname} ({size_mb:.1f} MB)")
    
    # 使用第一个视频
    VIDEO_PATH = f"input_videos/{existing_videos[0]}"
    VIDEO_NAME = existing_videos[0].replace('.mp4', '')
    OUTPUT_DIR = f"results/{VIDEO_NAME}"
    
    print(f"\n✓ 使用已上传视频: {existing_videos[0]}")
    print(f"视频路径: {VIDEO_PATH}")
    print(f"输出目录: {OUTPUT_DIR}")
    print("\n如需上传新视频，请先删除旧视频：!rm -rf /content/Track4World/input_videos/*")
else:
    print("未检测到已上传视频，请上传新视频：")
    uploaded = files.upload()

    for fname in uploaded:
        dst = f"input_videos/{fname}"
        with open(dst, "wb") as f:
            f.write(uploaded[fname])
        print(f"✓ 已保存: {dst} ({os.path.getsize(dst)/1e6:.1f} MB)")

    VIDEO_PATH = f"input_videos/{list(uploaded.keys())[0]}"
    VIDEO_NAME = list(uploaded.keys())[0].replace('.mp4', '')
    OUTPUT_DIR = f"results/{VIDEO_NAME}"

    print(f"\n视频路径: {VIDEO_PATH}")
    print(f"输出目录: {OUTPUT_DIR}")

In [ ]:
# Cell 5: DINO + SAM2 分割 (关键步骤！)
import os, sys

os.chdir("/content/Track4World")

# 检查是否已上传视频
if 'OUTPUT_DIR' not in globals():
    print("❌ 错误：请先运行 Cell 4 上传视频")
    raise RuntimeError("请先运行 Cell 4")

# ============ 可修改参数 ============
TEXT_PROMPT = "person. golf ball. golf club."
# ====================================

print(f"正在分割动态物体: {TEXT_PROMPT}")
print(f"输出目录: {OUTPUT_DIR}\n")

# 在bash命令中设置PYTHONPATH
cmd = f"""
cd /content/Track4World && \
export PYTHONPATH=/content/Track4World/submodules:$PYTHONPATH && \
python scripts/run_dino_sam2.py \
    --video-path {VIDEO_PATH} \
    --sam2-checkpoint checkpoints/sam2.1_hiera_large.pt \
    --output-dir {OUTPUT_DIR} \
    --text-prompt "{TEXT_PROMPT}"
"""

print(f"执行命令:\n{cmd}\n")
!{cmd}

# 验证mask生成
mask_dir = f"{OUTPUT_DIR}/mask"
if os.path.exists(mask_dir):
    mask_count = len([f for f in os.listdir(mask_dir) if f.endswith('.png')])
    print(f"\n✓ 分割完成！生成了 {mask_count} 个mask文件")
else:
    print("\n⚠️ 警告：未找到mask目录，可能分割失败")

In [ ]:
# Cell 6: Track4World 3d_efep 推理 (世界坐标系)

import os
os.chdir("/content/Track4World")

# ============ 可修改参数 ============
IMAGE_SIZE = 448
MAX_FRAMES = 20  # 测试用20帧，完整推理改为-1
# ====================================

print("=== Track4World 3d_efep 推理 ===")
print(f"坐标系: world_depthanythingv3")
print(f"模型: DA3")
print(f"分辨率: {IMAGE_SIZE}")
print(f"帧数: {MAX_FRAMES if MAX_FRAMES > 0 else '全部'}\n")

cmd = f"""
cd /content/Track4World && \
pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python && \
pip install numpy==1.26.4 opencv-python==4.8.1.78 && \
find /usr/local/lib/python3.12/dist-packages/cv2 -name "*.pyc" -delete && \
export PYTHONPATH=/content/Track4World:/content/Track4World/submodules:/content/utils3d:$PYTHONPATH && \
python3 -c "import sys; sys.path.insert(0, '/usr/local/lib/python3.12/dist-packages'); import cv2; print('cv2 OK')" && \
python3 demo.py \
    --mp4_path {VIDEO_PATH} \
    --coordinate world_depthanythingv3 \
    --mode 3d_efep \
    --Ts {MAX_FRAMES} \
    --ckpt_init checkpoints/track4world_da3.pth \
    --image_size {IMAGE_SIZE} \
    --save_base_dir {OUTPUT_DIR}
"""

print(f"执行命令:\n{cmd}\n")
!{cmd}

# 验证输出
output_3d = f"{OUTPUT_DIR}/3d_efep_output"
if os.path.exists(output_3d):
    ply_count = len([f for f in os.listdir(output_3d) if f.endswith('.ply')])
    print(f"\n✓ 推理完成！生成了 {ply_count} 个点云文件")
    
    key_files = ['trajectory_all_pointmap.npy', 'c2w.npy']
    for f in key_files:
        path = f"{output_3d}/{f}"
        if os.path.exists(path):
            print(f"  ✓ {f}")
        else:
            print(f"  ✗ {f} (缺失)")
else:
    print("\n⚠️ 警告：未找到输出目录")

In [ ]:
# Cell 6: Track4World 3d_efep 推理 (世界坐标系)

import os
os.chdir("/content/Track4World")

# ============ 可修改参数 ============
IMAGE_SIZE = 448
MAX_FRAMES = 20  # 测试用20帧，完整推理改为-1
# ====================================

print("=== Track4World 3d_efep 推理 ===")
print(f"坐标系: world_depthanythingv3")
print(f"模型: DA3")
print(f"分辨率: {IMAGE_SIZE}")
print(f"帧数: {MAX_FRAMES if MAX_FRAMES > 0 else '全部'}\n")

cmd = f"""
python demo.py \
    --mp4_path {VIDEO_PATH} \
    --coordinate world_depthanythingv3 \
    --mode 3d_efep \
    --Ts {MAX_FRAMES} \
    --ckpt_init checkpoints/track4world_da3.pth \
    --image_size {IMAGE_SIZE} \
    --save_base_dir {OUTPUT_DIR}
"""

print(f"执行命令:\n{cmd}\n")
!{cmd}

# 验证输出
output_3d = f"{OUTPUT_DIR}/3d_efep_output"
if os.path.exists(output_3d):
    ply_count = len([f for f in os.listdir(output_3d) if f.endswith('.ply')])
    print(f"\n✓ 推理完成！生成了 {ply_count} 个点云文件")
    
    # 检查关键文件
    key_files = ['trajectory_all_pointmap.npy', 'c2w.npy']
    for f in key_files:
        path = f"{output_3d}/{f}"
        if os.path.exists(path):
            print(f"  ✓ {f}")
        else:
            print(f"  ✗ {f} (缺失)")
else:
    print("\n⚠️ 警告：未找到输出目录")

In [ ]:
# Cell 7: 打包并下载结果

import os
from google.colab import files

os.chdir("/content/Track4World")

zip_name = f"/content/{VIDEO_NAME}_official_results.zip"

print(f"正在打包: {OUTPUT_DIR}")
print("排除: input_copy.mp4 (节省空间)\n")

!zip -r {zip_name} {OUTPUT_DIR} -x "*/input_copy.mp4"

if os.path.exists(zip_name):
    size_mb = os.path.getsize(zip_name) / 1e6
    print(f"\n✓ 压缩完成: {size_mb:.1f} MB")
    print("正在下载...")
    files.download(zip_name)
    print("\n下载已启动，请查看浏览器下载列表")
else:
    print("\n✗ 打包失败")

## 本地可视化说明

下载结果后，在本地使用以下命令可视化：

```bash
# 解压结果
unzip *_official_results.zip

# 使用世界坐标系可视化 (前景-背景分离)
cd E:/bishe2/Track4World
E:/Conda/envs/track4world/python.exe visualization/vis_3d_efep_world.py \
    --ply_dir ../results/your_video_name/3d_efep_output \
    --save_dir ../recordings/your_video_name

# 浏览器打开: http://localhost:8080
```

**关键特性**:
- 静态背景保持不动
- 动态物体有彩色轨迹
- 可调节轨迹长度、点云大小等参数